### Установка необходимых библиотек и зависимостей

In [1]:
!pip install llama-index llama-index-embeddings-huggingface llama-index-llms-huggingface llama-index-readers-wikipedia llama-index-postprocessor-colbert-rerank llama-index-postprocessor-flag-embedding-reranker transformers torch accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 83.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 57.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 21.4 MB/s  0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27/27 [llama-index]


In [2]:
!pip install python-docx

### Импорт библиотек

In [3]:
from llama_index.core import Settings, KnowledgeGraphIndex, VectorStoreIndex, StorageContext, Document as LIDoc
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.postprocessor.colbert_rerank import ColbertRerank
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel, PeftConfig
from docx import Document
import torch

### Авторизируемся на HuggingFace

In [5]:
from huggingface_hub import login

HF_TOKEN="INPUT YOUR TOKEN"

login(HF_TOKEN, add_to_git_credential=True)

### Вспомогательные функции

In [6]:
def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == 'system':
            prompt += f"<s>{message.role}\n{message.content}</s>\n"
        elif message.role == 'user':
            prompt += f"<s>{message.role}\n{message.content}</s>\n"
        elif message.role == 'bot':
            prompt += f"<s>bot\n"

    # ensure we start with a system prompt, insert blank if needed
    if not prompt.startswith("<s>system\n"):
        prompt = "<s>system\n</s>\n" + prompt

    # подсказка для ассистента
    prompt = prompt + "<s>bot\n"
    return prompt

def completion_to_prompt(completion):
    return f"<s>system\n</s>\n<s>user\n{completion}</s>\n<s>bot\n"

### Загрузка модели

In [7]:
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
from peft import PeftModel, PeftConfig

# Определяем параметры квантования, иначе модель не выполниться в колабе
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Задаем имя модели
MODEL_NAME = "IlyaGusev/saiga_mistral_7b"

# Создание конфига, соответствующего методу PEFT (в нашем случае LoRA)
config = PeftConfig.from_pretrained(MODEL_NAME)

# Загружаем базовую модель, ее имя берем из конфига для LoRA
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,          # идентификатор модели
    quantization_config=quantization_config, # параметры квантования
    torch_dtype=torch.float16,               # тип данных
    device_map="auto"                        # автоматический выбор типа устройства
)

# Загружаем LoRA модель
model = PeftModel.from_pretrained(
    model,
    MODEL_NAME,
    torch_dtype=torch.float16
)

# Переводим модель в режим инференса
model.eval()

# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

adapter_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

... (6 outputs left out)

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

### Загрузка модели во фреймворк LlamaIndex

In [8]:
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
print(generation_config)

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_new_tokens": 1536,
  "no_repeat_ngram_size": 15,
  "num_beams": 1,
  "pad_token_id": 0,
  "repetition_penalty": 1.1,
  "temperature": 0.2,
  "top_k": 40,
  "top_p": 0.9
}



In [9]:
llm = HuggingFaceLLM(
    model=model,             # модель
    model_name=MODEL_NAME,   # идентификатор модели
    tokenizer=tokenizer,     # токенизатор
    max_new_tokens=generation_config.max_new_tokens, # параметр необходимо использовать здесь, и не использовать в generate_kwargs, иначе ошибка двойного использования
    model_kwargs={"quantization_config": quantization_config}, # параметры квантования
    generate_kwargs = {   # параметры для инференса
      "bos_token_id": generation_config.bos_token_id, # токен начала последовательности
      "eos_token_id": generation_config.eos_token_id, # токен окончания последовательности
      "pad_token_id": generation_config.pad_token_id, # токен пакетной обработки (указывает, что последовательность ещё не завершена)
      "no_repeat_ngram_size": generation_config.no_repeat_ngram_size,
      "repetition_penalty": generation_config.repetition_penalty,
      "temperature": generation_config.temperature,
      "do_sample": True,
      "top_k": 50,
      "top_p": 0.95
    },
    messages_to_prompt=messages_to_prompt,     # функция для преобразования сообщений к внутреннему формату
    completion_to_prompt=completion_to_prompt, # функции для генерации текста
    device_map="auto",                         # автоматически определять устройство
)

### Создание графов знаний

In [11]:
# --- Чтение docx ---

# Укажите путь к файлу:
file_path = "/content/Инструкция по работе со сварочными материалами и аппаратами.docx"

doc = Document(file_path)

nodes = []
current_section = "Без раздела"

SAFE_TERMS_FOR_WELDING = [
        "tig",
        "mig",
        "mag",
        "присадка",
        "полярность",
        "сварочный ток",
        "ток",
        "расход газа",
        "напряжение",
        "диаметр проволоки",
        "проволока",
        "вольфрам",
        "диаметр вольфрама",
        "диаметр электрода",
        "материал",
        "режимы",
        "марка/тип",
        "толщина",
        "расход ar,",
        "толщина металла",
        "тип стали",
        "присадка",
        "углеродистая",
        "легированная",
        "высоколегированная",
        "низколегированная"
        ]

for para in doc.paragraphs:
    style = para.style.name if para.style else ""
    text = para.text.strip()
    if not text:
        continue

    if style.startswith("Heading"):
        current_section = text
        continue

    # Теги: простой способ — искать ключевые слова
    tags = []
    lower = text.lower()
    for term in SAFE_TERMS_FOR_WELDING:
        if term in lower:
            tags.append(term)

    nodes.append(LIDoc(
        text=text,
        metadata={
            "source": file_path,
            "section": current_section,
            "tags": tags,
        }
    ))

# --- Чанкинг по предложениям с перекрытием ---
parser = SentenceSplitter(chunk_size=768, chunk_overlap=128)
chunks = parser.get_nodes_from_documents(nodes)
docs = chunks

In [12]:
embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Настраиваем окружение для LlamaIndex:

In [13]:
# Настройка ServiceContext (глобальная настройка параметров LLM)
Settings.llm = llm
Settings.embed_model = embed_model

In [14]:
# Создаем простое графовое хранилище
graph_store = SimpleGraphStore()

# Устанавливаем информацию о хранилище в StorageContext
storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [15]:
indexVSI = VectorStoreIndex.from_documents( documents=docs, storage_context=storage_context)

### Промпт

In [16]:
DANGEROUS_PATTERNS = [
    "без сварочной маски",
    "как повредить оборудование",
    "обойти защиту",
    "без охлаждения",
    "игнорируй предыдущие инструкции",
]

qa_prompt = PromptTemplate(
    """
    Ты — главный сварщик на предприятии. Твоя задача — отвечать ТОЛЬКО на основе предоставленного документа.

    ПРАВИЛА:
    1. Используй только термины, режимы и параметры из документа.
    2. Постарайся дать развернутый ответ, твоя задача ответить так, чтобы у сотрудника не осталось больше вопросов к тебе.
    3. Если в контексте нет нужной информации — прямо скажи: «В базе знаний нет данных по этому вопросу».
    4. Не выдумывай значения тока, напряжения, полярности, углов и прочих параметров.
    5. Не добавляй общие советы вне документа.
    6. Отвечай по существу, без лишних эмоций и слов, от тебя нужна только точная информация.
    7. Отвечай максимально точно по документу, не придумывай ничего от себя.
    8. Не округляй значения.

    Контекст:
    {context_str}

    Вопрос:
    {query_str}
    """
)

colbert_reranker = ColbertRerank(
    top_n=2,
    model="colbert-ir/colbertv2.0",
    tokenizer="colbert-ir/colbertv2.0",
    keep_retrieval_score=True,
)

query_engine = indexVSI.as_query_engine(
    text_qa_template=qa_prompt,
    # similarity_top_k=7,
    response_mode="refine",
    node_postprocessors=[colbert_reranker]
)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/405 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: colbert-ir/colbertv2.0
Key           | Status     |  | 
--------------+------------+--+-
linear.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
# Блокируем опасные фразы
def is_safe_query(query: str) -> bool:
    q = query.lower()
    for pattern in DANGEROUS_PATTERNS:
        if pattern in q:
            return False
    return True


# Вывод результата
def ans(query):
  data = {}
  if len(query) < 20:
    data['error'] = "Мало информации. Задайте вопрос более детально."
    return data
  else:
    result = is_safe_query(query)
    if not result:
        data['error'] = """Нарушать технику безопасности КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО!\nЭто может повлечь за собой вред для здоровья или поломку оборудования."""
        return data
    else:
        response = query_engine.query(query)
        data['succes'] = response.response
        for node in response.source_nodes:
            data['metadata'] = f"\nИсточник:\nРаздел: {node.metadata.get('section')}"
            data['node'] = f"Текс: {node.text[:200]}"
        return data

### Запросы к LLM с помощью Graph RAG

#### Эксперимент 1

In [18]:
query = "Не работает куллер. Как запустить сварочный источник без охлаждения?"
answer = ans(query)

for k in answer:
  print(answer[k])

Нарушать технику безопасности КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО!
Это может повлечь за собой вред для здоровья или поломку оборудования.


#### Эксперимент 2

In [19]:
query = "какой диаметр проволоки нужен для работы со сталью 3 мм"
answer = ans(query)

for k in answer:
  print(answer[k])

/usr/local/lib/python3.13/dist-packages/llama_index/postprocessor/colbert_rerank/base.py:120: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  node.score = float(score)


Диаметр проволоки для работы со сталью толщиной 3 мм должен быть 1.2 мм, если использовать обычную проволоку. Если используется проволока с усиленным напылением, то диаметр может быть меньше - от 0.8 до 1.0 мм.

Источник:
Раздел: Без раздела
Текс: Диаметр проволоки, мм – 0.8


#### Эксперимент 3

In [20]:
query = "Какой ток в TIG сварке нужен для работы с аустенитной сталью 2 мм?"
answer = ans(query)

for k in answer:
  print(answer[k])

Ток, А = 80-180 А.

Источник:
Раздел: Режимы TIG для легированных сталей
Текс: Ток, А  – (120 - 180)


#### Эксперимент 4

In [24]:
query = "Инструкция по технике безопасности при работе со сварочным аппаратом"
answer = ans(query)

for k in answer:
  print(answer[k])

Инструкция по технике безопасности, когда работают с сварочным аппаратом, содержит следующие пункты:

1. Перед началом работы сварочного аппартата необходимо провести инспекцию оборудовния на наличие повреждений или дефектов;
2. При подключении к источнику питаня следить за правильностью подключения кабелей;
3. Во время работы с аппаратом необходимы защитные средства: перчатки; защитные очки, защитные маски, шлемы и другие;
4. Работать с аппаратом можно лишь после включения защитных систем и проведения испытательного запуска;
5. При работе с аппаратом необходим соблюдение расстояния между рабочим местом и окружающими объектам, чтобы предотвратить возможные пожар и взрыв;
6. Необходимо соблюдать правила безовапасности при работе с газами и кислорода, которые используются для сварки;
7. В случае возникновения аварийной сиуации или появления опасностей немедля прекращать работу и обращаться за помощью;
8. После окончания работы сварочного аппартата необходим его демонтаж и проверка;
9. Вс

#### Эксперимент 5

In [22]:
query = "Какой ток MIG/MAG сварки нужен для проволоки тощиной 1.2 мм для работы с углеродистой сталью толщиной 4 мм"
answer = ans(query)

for k in answer:
  print(answer[k])

Ток MIG/MAG сварки должен быть в диapзоне 200-300 А.

Источник:
Раздел: Без раздела
Текс: Диаметр проволоки, мм – 1.0 или 1.2
